In [4]:
import os
import zipfile
import tempfile
import geopandas as gpd
import pandas as pd

In [2]:
# Define base directory where all the "Compilation of GIS..." folders are
USER = '157412'
base_dir = f'C:\\Users\\{USER}\\OneDrive - UTS (1)\\ISF Projects - 23204_Quadrature_Mineral Availability for Renewables PEMMS\\3. Resources\\Graphite Production\\Sciencebase data'

# List of folders
folders = [
    "Compilation of GIS Africa",
    "Compilation of GIS China",
    "Compilation of GIS Indopacific",
    "Compilation of GIS Latin American and Carib",
    "Compilation of GIS SW Asia"
]

def get_gdb_path(folder_path):
    """Finds the .gdb or .gdb.zip in a folder."""
    for root, dirs, files in os.walk(folder_path):
        # Check for unzipped .gdb folders
        for d in dirs:
            if d.endswith('.gdb'):
                return os.path.join(root, d)
        # Check for zipped .gdb files
        for f in files:
            if f.endswith('.gdb.zip') or (f.endswith('.zip') and 'GDB' in f.upper()):
                return os.path.join(root, f)
    return None

print("Setup complete. Ready to scan folders.")

Setup complete. Ready to scan folders.


In [5]:
all_graphite_data = []

for folder_name in folders:
    folder_path = os.path.join(base_dir, folder_name)
    gdb_path = get_gdb_path(folder_path)
    
    if not gdb_path:
        print(f"Skipping {folder_name}: No GDB found.")
        continue
        
    print(f"--- Processing: {folder_name} ---")
    
    try:
        # 1. List all layers in the GDB to find the right one
        layers = gpd.list_layers(gdb_path)
        print(f"Available layers: {layers['name'].tolist()}")
        
        # 2. Usually, USGS datasets use 'Mineral_Facilities' or 'Exploration_Sectors'
        # We'll look for layers containing 'Mineral' or 'Facility' or 'Site'
        target_layer = None
        for l in layers['name']:
            if any(x in l for x in ['Facility', 'Site', 'Occurrence', 'Deposit']):
                target_layer = l
                break
        
        if target_layer:
            print(f"Loading layer: {target_layer}...")
            gdf = gpd.read_file(gdb_path, layer=target_layer)
            
            # 3. Filter for Graphite
            # We use case-insensitive search in common USGS columns like 'COMMODITY' or 'COMMOD_1'
            graphite_mask = gdf.astype(str).apply(lambda x: x.str.contains('graphite', case=False)).any(axis=1)
            graphite_gdf = gdf[graphite_mask].copy()
            
            if not graphite_gdf.empty:
                graphite_gdf['source_region'] = folder_name
                all_graphite_data.append(graphite_gdf)
                print(f"Found {len(graphite_gdf)} graphite entries.")
            else:
                print("No graphite projects found in this layer.")
                
    except Exception as e:
        print(f"Error loading {gdb_path}: {e}")

# Combine everything into one master GeoDataFrame
if all_graphite_data:
    master_gdf = pd.concat(all_graphite_data, ignore_index=True)
    print("\nSUCCESS: Master Graphite Dataset Created.")
else:
    print("\nNo graphite data was found in any folder.")

--- Processing: Compilation of GIS Africa ---


c:\Users\157412\.conda\envs\pemmss\Lib\site-packages\pyogrio\core.py:130: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Point' is converted to 'Point Z'
  return ogr_list_layers(get_vsi_path_or_buffer(path_or_buffer))
c:\Users\157412\.conda\envs\pemmss\Lib\site-packages\pyogrio\core.py:130: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiLineString' is converted to 'MultiLineString Z'
  return ogr_list_layers(get_vsi_path_or_buffer(path_or_buffer))
c:\Users\157412\.conda\envs\pemmss\Lib\site-packages\pyogrio\raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Point' is converted to 'Point Z'
  return ogr_read(


Available layers: ['AFR_Mineral_Facilities', 'AFR_Mineral_Deposits', 'AFR_Mineral_Exploration', 'AFR_Mineral_Resources_Coal', 'AFR_Mineral_Resources_Copper', 'AFR_Mineral_Resources_Gabon', 'AFR_Mineral_Resources_Mauritania', 'AFR_Mineral_Resources_PGE', 'AFR_Mineral_Resources_Potash', 'AFR_Infra_OG_Pipelines', 'AFR_Infra_Power_Stations', 'AFR_OG_Provinces_Continuous', 'AFR_OG_Provinces_Conventional', 'AFR_OG_Resources_Recoverable', 'AFR_Infra_Transport_Ports', 'AFR_Infra_Transport_Rail', 'AFR_Political_Cities', 'AFR_NaturalFeatures_Rivers', 'AFR_NaturalFeatures_Lakes', 'AFR_Political_ADM0_Boundaries', 'AFR_Political_ADM1_Boundaries', 'AFR_Infra_OG_LNG_Terminals', 'AFR_Infra_Power_Transmission', 'AFR_Infra_Transport_Road']
Loading layer: AFR_Mineral_Deposits...
Found 22 graphite entries.
--- Processing: Compilation of GIS China ---
Available layers: ['CHN_Mineral_Resources_Antimony', 'CHN_Mineral_Resources_Coal', 'CHN_Mineral_Resources_Copper', 'CHN_Mineral_Resources_Phosphate', 'CHN_Mi

c:\Users\157412\.conda\envs\pemmss\Lib\site-packages\pyogrio\core.py:130: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Point' is converted to 'Point Z'
  return ogr_list_layers(get_vsi_path_or_buffer(path_or_buffer))
C:\Users\157412\AppData\Local\Temp\ipykernel_44328\4268447666.py:47: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master_gdf = pd.concat(all_graphite_data, ignore_index=True)
